# LiftLab — Experiment Walk-through

This notebook walks the full LiftLab pipeline end-to-end on the GA4 Merchandise Store sample:

1. Raw data overview
2. Cleaning + feature engineering
3. Experiment assignment + SRM pre-flight
4. Treatment-effect simulation (known ground truth)
5. Frequentist + Bayesian inference (user level)
6. CUPED variance reduction
7. Segmentation
8. Guardrails + final decision



## Setup

Make `src/` importable, then pull in every LiftLab module.

In [ ]:
import sys
from pathlib import Path

SRC = Path.cwd().parent / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd

from clean import build_clean_sessions
from assign_experiment import assign_users, DEFAULT_EXPERIMENT, get_assignment_summary
from simulate_treatment import simulate_treatment_effects, get_ground_truth
from frequentist import run_all_tests, srm_check
from bayesian import run_all_bayesian
from cuped import run_all_cuped
from segmentation import run_all_segmentation
from guardrails import run_all_guardrails
from recommendation import make_recommendation

## 1. Raw data overview

The dataset is `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*` — Google Merchandise Store traffic, Nov 2020 through Jan 2021, flattened from nested GA4 events to one row per session via `sql/flatten_sessions.sql`.

November serves as the pre-period for CUPED covariates; the experiment runs over December–January.

In [ ]:
raw = pd.read_csv(Path.cwd().parent / "data" / "sessions_raw.csv")
print(f"Raw sessions: {len(raw):,}")
print(f"Unique users: {raw['user_pseudo_id'].nunique():,}")
raw.head(3)

## 2. Cleaning + feature engineering (`src/clean.py`)

Standardizes types, fills nulls, drops duplicates, and engineers:
- `converted` — binary flag from `transactions > 0`
- `session_duration_sec` — `(end - start) / 1e6`, clipped to ≥0
- `is_bounce_proxy` — `pageviews ≤ 1 AND duration < 10s`
- `engagement_score` — weighted log of pageviews / events / duration
- `prior_*` — leakage-safe cumulative-minus-current-row covariates per user (for CUPED)

In [ ]:
df = build_clean_sessions()
print(f"Cleaned sessions: {len(df):,}")
print(f"Conversion rate (session-level): {df['converted'].mean():.4f}")
print(f"Avg session duration: {df['session_duration_sec'].mean():.1f}s")
df.head(3)

## 3. Experiment assignment + SRM (`src/assign_experiment.py`, `src/frequentist.py`)

Users are assigned 50/50 control/treatment with stratification by `device_type` and seed 42 for reproducibility. Assignment is at the USER level — a single user sees the same variant across all sessions.

Immediately after assignment, we run an SRM (Sample Ratio Mismatch) chi-squared check. If the variant counts don't match the configured split, every downstream result is suspect — so SRM is treated as a pre-flight gate, not a footnote.

In [ ]:
df = assign_users(df)
summary = get_assignment_summary(df)
print(f"Total users: {summary['total_users']:,} | Sessions: {summary['total_sessions']:,}")
print(f"Date range: {summary['date_range']}")
print(f"Variant users: {summary['variant_users']}")

In [ ]:
srm = srm_check(df)
print(f"SRM: {'PASS' if srm['passed'] else 'FAIL'}  "
      f"chi2={srm['chi2']:.4f}  p={srm['p_value']:.4f}  "
      f"actual split={srm['actual_treatment_share']:.4f}")

## 4. Treatment simulation (`src/simulate_treatment.py`)

Because this isn't a real experiment, we inject synthetic effects with known ground truth so the stats engine has something to recover. The treatment is meant to evoke a checkout-flow redesign:

- **+3% conversion** globally, **+5% on mobile** (compounded — see `CLAUDE.md` quirks)
- **+2% revenue** globally
- **+2% add-to-cart** globally
- **−3% pageviews on desktop**
- **+5s session duration** (additive)

Continuous effects get per-row Gaussian noise so the treatment group doesn't look uniformly shifted.

In [ ]:
df = simulate_treatment_effects(df)
print("Variant means after simulation:")
df.groupby("variant")[["converted", "revenue", "pageviews", "add_to_cart_events", "session_duration_sec"]].mean()

In [ ]:
import pprint
pprint.pprint(get_ground_truth())

## 5. Frequentist + Bayesian inference (`src/frequentist.py`, `src/bayesian.py`)

Both modules aggregate to the user level before testing — sessions within a user are correlated, so session-level tests inflate apparent significance.

- **Frequentist** runs a two-proportion z-test on binary outcomes and Welch's t-test on continuous outcomes, with analytical CIs and a chunked bootstrap.
- **Bayesian** uses Beta-Binomial (Beta(1,1) prior) on binary and a Normal posterior on the mean for continuous, with 100K samples.

Where they disagree, the disagreement is informative.

In [ ]:
freq = run_all_tests(df)
bayes = run_all_bayesian(df)

freq_df = pd.DataFrame(freq["primary"])
bayes_df = pd.DataFrame(bayes["primary"])[["metric_name", "prob_treatment_better", "expected_loss_ship", "expected_loss_no_ship", "ship_recommended"]]
freq_df.merge(bayes_df, on="metric_name")[
    ["metric_name", "relative_lift", "p_value", "significant", "prob_treatment_better", "ship_recommended"]
]

## 6. CUPED variance reduction (`src/cuped.py`)

CUPED adjusts each user's outcome by their pre-experiment behavior to subtract predictable noise: `Y_adj = Y − (X − X̄)θ`, θ fit on combined data. Covariates: `prior_sessions`, `prior_revenue`, `prior_avg_engagement`.

On this GA4 sample the reduction is small (0–2%) because most users in the experiment window are new and have no pre-period — `prior_*` columns are mostly zero. The implementation is correct; the limit is the data. Production-scale longitudinal data would yield the textbook 20–50% reductions.

In [ ]:
cuped = run_all_cuped(df)
pd.DataFrame(cuped["results"])[[
    "metric_name", "raw_lift", "raw_p_value", "adj_lift", "adj_p_value",
    "variance_reduction", "ci_width_improvement",
]]

## 7. Segmentation (`src/segmentation.py`)

Heterogeneous-effect search: run the primary-metric tests within each value of device, top-10 country, traffic source, traffic medium. Min 100 users per variant per segment; below that, flagged underpowered.

In [ ]:
seg = run_all_segmentation(df)
rows = []
for col, results in seg["segments"].items():
    for s in results:
        for m in s["metrics"]:
            if m["metric_name"] == "conversion_rate":
                rows.append({
                    "segment": f"{col}={s['segment_value']}",
                    "n_ctrl": s["n_users_control"],
                    "n_treat": s["n_users_treatment"],
                    "lift": m["relative_lift"],
                    "p": m["p_value"],
                    "significant": m["significant"],
                })
pd.DataFrame(rows).sort_values("lift", ascending=False).head(15)

## 8. Guardrails + final decision (`src/guardrails.py`, `src/recommendation.py`)

Guardrails are threshold-based and direction-aware: bounce_rate/volatility are `lower_is_better`, session_depth is `higher_is_better`. The recommendation module runs every analysis and applies a deterministic decision tree (guardrails → REJECT → INVESTIGATE → SHIP → HOLD) to produce a final verdict.

In [ ]:
guards = run_all_guardrails(df)
print(f"Overall: {guards['overall_status'].upper()}")
pd.DataFrame(guards["results"])[[
    "name", "direction", "relative_diff", "warning_threshold", "fail_threshold", "status",
]]

In [ ]:
rec = make_recommendation(df)
print(rec["summary"])

## Takeaway

The treatment looks great on the topline — revenue per user is significantly up, Bayesian P(T>C) clears 0.99 — but the **revenue volatility** guardrail breached the +15% policy threshold. The platform's final verdict is therefore **HOLD**, not SHIP. That tension between "the headline numbers are good" and "the safety check failed" is the canonical scenario LiftLab is built to surface.